# 8주차 — DML — 입력·수정·삭제와 트랜잭션
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다.
어떤 셀이 오류로 막히거나 데이터가 엉켰다면, 아무 셀에서 `reset_db()`를 실행하면 처음 상태로 돌아갑니다.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, shutil, sqlite3, urllib.request
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/mini_emr/mini_emr.sqlite"
DB_PATH = "mini_emr.sqlite"
WORK_PATH = "mini_emr_practice.sqlite"   # 실습에 사용할 파일

con = None

def reset_db():
    """DB를 원본 상태로 되돌리고 다시 연결 (막히면 언제든 호출)"""
    global con
    if con is not None:
        con.close()
    try:
        urllib.request.urlretrieve(DB_URL, DB_PATH)
    except Exception as e:
        if not os.path.exists(DB_PATH):
            raise RuntimeError(
                "DB 다운로드 실패 — 왼쪽 파일 탭에 mini_emr.sqlite를 업로드한 뒤 이 셀을 다시 실행하세요."
            ) from e
        print("다운로드 실패 — 업로드되어 있는 파일을 사용합니다.")
    if WORK_PATH != DB_PATH:
        shutil.copyfile(DB_PATH, WORK_PATH)   # 원본 보존, 실습은 사본에서
    con = sqlite3.connect(WORK_PATH)
    con.execute("PRAGMA foreign_keys = ON")   # SQLite는 기본으로 꺼져 있음 — 반드시 켠다
    assert con.execute("PRAGMA foreign_keys").fetchone()[0] == 1

def q(sql):
    """조회(SELECT) 실행 → 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """데이터/구조를 바꾸는 SQL(INSERT/UPDATE/DELETE/CREATE 등) 실행"""
    con.executescript(sql); con.commit(); print("실행 완료")

reset_db()
q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 1. INSERT
이 노트북은 **사본 DB**에서 실습합니다. 아래 셀을 두 번 실행하면 PK 중복 에러가 납니다 — 기본키가 정상 동작하는 것입니다. `reset_db()` 후 다시 실행하세요.

In [ ]:
# 가상 환자 P051 등록 (재실행 시 PK 중복 에러 = 정상)
run("""
INSERT INTO patient (patient_id, sex, birth_date, blood_type)
VALUES ('P051', 'F', '1988-03-15', 'A+');
""")

In [ ]:
# 확인
q("""
SELECT * FROM patient WHERE patient_id = 'P051'
""")

In [ ]:
# FK가 실제로 켜져 있는지 확인 — 존재하지 않는 내원(V9999)에 처방을 넣어 본다
try:
    run("""
    INSERT INTO prescription (rx_id, visit_id, drug_code, drug_name, dose, duration_days)
    VALUES ('R9999', 'V9999', 'MET500', '메트포르민', '500mg', 30);
    """)
except Exception as e:
    print('거부됨 →', e)   # FOREIGN KEY constraint failed — 참조 무결성(2주차)

## 2. UPDATE — WHERE 확인 습관

In [ ]:
# ① 먼저 SELECT로 대상 확인
q("""
SELECT lab_id, test_name, result_value FROM lab WHERE lab_id = 'L0044'
""")

In [ ]:
# ② 그 다음 UPDATE
run("""
UPDATE lab SET result_value = 176 WHERE lab_id = 'L0044';
""")

In [ ]:
# ③ 결과 재확인
q("""
SELECT lab_id, test_name, result_value FROM lab WHERE lab_id = 'L0044'
""")

## 3. 트랜잭션 — 없던 일로 만들기

In [ ]:
# BEGIN → UPDATE → ROLLBACK: WHERE 절이 없는 UPDATE의 영향을 확인하고 되돌린다
cur = con.cursor()
cur.execute("BEGIN")
cur.execute("UPDATE lab SET result_value = 0")
print('UPDATE 직후:', cur.execute("SELECT COUNT(*) FROM lab WHERE result_value = 0").fetchone())
con.rollback()
print('롤백 후   :', cur.execute("SELECT COUNT(*) FROM lab WHERE result_value = 0").fetchone())

## 4. soft delete + 감사로그 패턴

In [ ]:
# 감사로그 테이블과 정정 이력 남기기 (재실행하면 이력이 한 건씩 쌓임 — reset_db()로 초기화)
run("""
CREATE TABLE IF NOT EXISTS lab_audit (
  audit_id INTEGER PRIMARY KEY,
  lab_id TEXT, old_value REAL, new_value REAL,
  changed_by TEXT, changed_at DATETIME, reason TEXT
);
INSERT INTO lab_audit (lab_id, old_value, new_value, changed_by, changed_at, reason)
VALUES ('L0044', 167, 176, 'S001', '2026-04-02 09:30', '재검 결과 반영');
""")

In [ ]:
# 감사로그 확인
q("""
SELECT * FROM lab_audit
""")

## 연습문제
Q1·Q2·Q4는 데이터를 바꾸는 문제이므로 `run()` 안에, 결과 확인은 별도 `q()` 셀에서 하세요.

In [ ]:
# [연습 Q1] P051의 외래 내원 한 건을 visit에 INSERT 하라 (visit_id는 'V0101' 등 새 값)
run("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q2] P051의 혈액형을 B+로 정정하라 — 먼저 q()로 SELECT 확인 후, 이 셀에서 UPDATE
run("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q3] WHERE 없는 UPDATE를 실행해 피해 규모를 q()로 확인한 뒤, reset_db()로 복구하라
run("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q4] lab_audit에 '단위 오입력 정정' 이력을 한 건 더 남겨 보라
run("""
-- 여기에 SQL을 작성하세요

""")

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**A1. P051 내원**
```sql
INSERT INTO visit (visit_id, patient_id, visit_date, department, visit_type)
VALUES ('V0101', 'P051', '2026-04-01', '가정의학과', '외래');
```

**A2. 혈액형 정정**
```sql
SELECT * FROM patient WHERE patient_id='P051';
UPDATE patient SET blood_type='B+' WHERE patient_id='P051';
SELECT * FROM patient WHERE patient_id='P051';
```